# Delivery Orders Clustering Analysis
### K-Means vs. DBSCAN

**Objective:** Explore delivery-order patterns using two numerical features, compare K-Means and DBSCAN, and explain the results.

**Dataset:** `delivery_operations_students(1).csv`

> Keep the CSV file in the same folder as this notebook when running it locally or on GitHub/Google Colab. The analysis uses `distance_km` and `delivery_time_min` as the two clustering features. `order_id` is an identifier, not a model feature.

## 1. Import libraries and load the data

We use pandas for data handling, Matplotlib/Seaborn for visualization, and scikit-learn for clustering and feature scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# The CSV should be in the same folder as this notebook.
file_path = "delivery_operations_students(1).csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumn information:")
df.info()
print("\\nMissing values per column:")
display(df.isna().sum())

## 2. Prepare the two features

We will cluster orders using:

- **Distance (km):** how far the delivery travels.
- **Delivery time (minutes):** total delivery time.

Clustering is distance-based, so the features are standardized before fitting the models. This prevents a feature's numeric scale from dominating the distance calculation.

In [ ]:
features = ["distance_km", "delivery_time_min"]
X = df[features].copy()

print("Missing values in selected features:")
display(X.isna().sum())

# Drop rows only if the selected features contain missing values.
valid_rows = X.dropna().index
df_model = df.loc[valid_rows].copy()
X = X.loc[valid_rows].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Rows used for clustering:", len(X))

## 3. Visualizations before applying any models

These plots help us understand the feature distributions, relationship, and possible groups/outliers before clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=df_model, x="distance_km", y="delivery_time_min",
    ax=axes[0], s=55
)
axes[0].set_title("Delivery Distance vs. Delivery Time")
axes[0].set_xlabel("Distance (km)")
axes[0].set_ylabel("Delivery time (min)")

sns.histplot(data=df_model, x="distance_km", kde=True, ax=axes[1])
axes[1].set_title("Distribution of Delivery Distance")
axes[1].set_xlabel("Distance (km)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df_model, x="delivery_time_min", kde=True, ax=axes[0])
axes[0].set_title("Distribution of Delivery Time")
axes[0].set_xlabel("Delivery time (min)")

sns.heatmap(
    df_model[features].corr(), annot=True, cmap="coolwarm",
    vmin=-1, vmax=1, ax=axes[1]
)
axes[1].set_title("Correlation Between the Two Features")

plt.tight_layout()
plt.show()

### Initial visual interpretation

- The scatter plot shows whether delivery distance and delivery time form visible groups or contain unusual observations.
- Histograms show the spread and skew of each feature.
- Correlation describes the strength and direction of a **linear** relationship; it does not prove that one variable causes the other.
- We will use the model outputs below to check whether the apparent patterns form useful clusters.

## 4. K-Means clustering (using 2 features)

K-Means assigns every order to one of *k* clusters. It works best when groups are reasonably compact and separated. Since it requires the number of clusters in advance, we first inspect inertia across different values of *k*.

In [ ]:
inertias = []
k_values = range(1, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xticks(list(k_values))
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("Elbow Method for K-Means")
plt.show()

### Elbow and inertia — how to read the plot

**Inertia** is the sum of squared distances from each point to its assigned cluster center. It always decreases or stays the same as *k* increases, because adding clusters gives K-Means more flexibility.

Choose a value near the **elbow**: the point where the reduction in inertia begins to slow. The elbow is a visual guideline, not a guaranteed mathematically unique answer. Consider the plot and whether the clusters are interpretable.

In [ ]:
# Set this after inspecting the elbow plot. Change it if another k looks more appropriate.
K = 3

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df_model["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

print("K-Means cluster counts:")
kmeans_counts = (
    df_model["kmeans_cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="order_count")
)
display(kmeans_counts)

In [ ]:
# Visualize K-Means assignments in the original units.
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_model, x="distance_km", y="delivery_time_min",
    hue="kmeans_cluster", palette="tab10", s=65
)
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(
    centers_original[:, 0], centers_original[:, 1],
    marker="X", s=250, c="black", label="Centroids"
)
plt.title(f"K-Means Clusters (k={K})")
plt.xlabel("Distance (km)")
plt.ylabel("Delivery time (min)")
plt.legend(title="Cluster")
plt.show()

### Explain the K-Means results

- **Cluster counts** show how many orders were assigned to each group. K-Means assigns every usable order to exactly one cluster, so the counts should sum to the number of rows used.
- **Centroids** are the average feature values for each cluster. A centroid with larger distance and longer delivery time represents a group of longer-distance/slower orders.
- Cluster labels such as `0`, `1`, and `2` are arbitrary identifiers, not rankings.
- If the scatter plot shows overlap, K-Means still assigns each order to a cluster; assignment alone does not mean the groups are naturally well separated.

In [ ]:
# Summarize the typical characteristics of each K-Means cluster.
kmeans_profile = (
    df_model.groupby("kmeans_cluster")[features]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)
display(kmeans_profile)

## 5. DBSCAN clustering (using the same 2 features)

DBSCAN groups points that are sufficiently dense. Unlike K-Means, it does not require a preset number of clusters and can mark isolated observations as **noise** (`-1`).

- `eps`: maximum neighborhood distance between points (in standardized feature space here).
- `min_samples`: number of points in an `eps` neighborhood required for a point to qualify as a core point (including the point itself).
- **Core point:** has at least `min_samples` points in its neighborhood.
- **Border point:** is not a core point but belongs to the neighborhood of a core point.
- **Noise point:** is not assigned to any cluster.

In [ ]:
# Initial DBSCAN settings. Since X_scaled is standardized, eps is measured
# in standardized feature-space units, not km or minutes.
EPS = 0.5
MIN_SAMPLES = 5

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
db_labels = dbscan.fit_predict(X_scaled)
df_model["dbscan_cluster"] = db_labels

# DBSCAN marks core points explicitly through core_sample_indices_.
core_mask = np.zeros(len(df_model), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True

# Non-core points with a cluster label are border points; label -1 means noise.
labels = df_model["dbscan_cluster"].to_numpy()
core_count = int(core_mask.sum())
noise_count = int((labels == -1).sum())
border_count = int(((~core_mask) & (labels != -1)).sum())

print(f"Core points:   {core_count}")
print(f"Border points: {border_count}")
print(f"Noise points:  {noise_count}")
print(f"Total rows:    {len(df_model)}")

In [ ]:
# Count orders in each DBSCAN label; -1 means noise.
dbscan_counts = (
    df_model["dbscan_cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="order_count")
)
display(dbscan_counts)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_model, x="distance_km", y="delivery_time_min",
    hue="dbscan_cluster", palette="tab10", s=65
)
plt.title(f"DBSCAN Clusters (eps={EPS}, min_samples={MIN_SAMPLES}; -1 = noise)")
plt.xlabel("Distance (km)")
plt.ylabel("Delivery time (min)")
plt.legend(title="Cluster label")
plt.show()

## 6. Effect of DBSCAN hyperparameters (`eps`, `min_samples`)

The next experiment compares several parameter combinations. Results depend on the scale of the data, which is why we use the standardized features.

In [ ]:
parameter_rows = []

for eps in [0.3, 0.5, 0.7, 1.0, 1.3]:
    for min_samples in [3, 5, 8]:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels_try = model.fit_predict(X_scaled)
        n_clusters = len(set(labels_try) - {-1})
        n_noise = int(np.sum(labels_try == -1))
        parameter_rows.append({
            "eps": eps,
            "min_samples": min_samples,
            "clusters_found": n_clusters,
            "noise_points": n_noise,
            "noise_percent": round(100 * n_noise / len(labels_try), 1)
        })

parameter_results = pd.DataFrame(parameter_rows)
display(parameter_results.sort_values(["eps", "min_samples"]).reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=parameter_results, x="eps", y="clusters_found",
    hue="min_samples", marker="o", ax=axes[0]
)
axes[0].set_title("Effect of Parameters on Number of Clusters")
axes[0].set_ylabel("Clusters found (excluding noise)")

sns.lineplot(
    data=parameter_results, x="eps", y="noise_percent",
    hue="min_samples", marker="o", ax=axes[1]
)
axes[1].set_title("Effect of Parameters on Noise")
axes[1].set_ylabel("Noise points (%)")

plt.tight_layout()
plt.show()

### How to explain the hyperparameter effects

- **Increasing `eps`** makes each point's neighborhood larger. This often connects more points, reduces the number of noise points, and may merge clusters. If `eps` becomes too large, distinct groups may collapse into one.
- **Decreasing `eps`** creates stricter neighborhoods. It can split groups and classify more observations as noise.
- **Increasing `min_samples`** makes the density requirement stricter, which often increases noise and can reduce the number of clusters.
- **Decreasing `min_samples`** makes it easier for dense regions to form clusters, but can also identify small or less meaningful groups.
- The effects are data-dependent, so use the parameter table and plots as evidence rather than assuming every change always behaves the same way.

## 7. Final comparison: K-Means vs. DBSCAN

| Aspect | K-Means | DBSCAN |
|---|---|---|
| Number of clusters | Must choose `k` in advance | Determined from density |
| Noise/outliers | Every order is assigned to a cluster | Can label orders as noise (`-1`) |
| Cluster shape | Best for compact, roughly spherical groups | Can find irregularly shaped dense groups |
| Main parameters | `k`, initialization | `eps`, `min_samples` |
| Sensitivity | Sensitive to outliers and initial centers | Sensitive to `eps`, `min_samples`, and feature scaling |
| Interpretability | Centroids are easy to summarize | Density groups and noise are useful for anomaly exploration |

### Conclusion template

Use the observed outputs from this notebook to complete the conclusion:

1. **K-Means:** `K` clusters were requested. Report the cluster counts and describe each cluster using its average distance and delivery time.
2. **DBSCAN:** Report the number of clusters excluding noise, plus core, border, and noise counts. Explain whether the chosen parameters produced meaningful groups or marked many orders as noise.
3. **Which is more suitable?** Prefer K-Means if the goal is to segment all orders into a chosen number of understandable groups. Prefer DBSCAN if identifying dense groups and potential unusual orders is important and the density structure is clear.
4. **Important limitation:** this analysis uses only distance and delivery time. Adding preparation time or order value could reveal different patterns, and clusters should be validated with domain knowledge before business decisions are made.

In [ ]:
# Compact comparison summary generated from the actual model outputs.
kmeans_cluster_count = df_model["kmeans_cluster"].nunique()
dbscan_cluster_count = len(set(df_model["dbscan_cluster"]) - {-1})
dbscan_noise_count = int((df_model["dbscan_cluster"] == -1).sum())

print(f"Rows analyzed: {len(df_model)}")
print(f"K-Means: {kmeans_cluster_count} clusters; all rows assigned.")
print(f"DBSCAN: {dbscan_cluster_count} clusters; {dbscan_noise_count} noise points.")
print("\\nK-Means cluster counts:")
display(df_model["kmeans_cluster"].value_counts().sort_index().rename("orders").to_frame())
print("DBSCAN labels (-1 indicates noise):")
display(df_model["dbscan_cluster"].value_counts().sort_index().rename("orders").to_frame())

## 8. Save the results (optional)

This cell saves the cluster labels alongside the original dataset for later review. The output file is created in the notebook's current working directory.

In [ ]:
results = df_model.copy()
results.to_csv("delivery_orders_with_cluster_labels.csv", index=False)
print("Saved: delivery_orders_with_cluster_labels.csv")